# 128→64 DWT：別患者ファインチューニング（30,000 epoch）

事前学習済みの128→64 DWTモデルを読み込み、`Data/TestData/<pair>/` 内の各ペアの2つの `.npz` を画像プールとして使い、256→128版と同じ合成DVFカリキュラムで30,000 epoch学習します。各 `.npz` は `Train` キーを持つ必要があります。

In [ ]:
from pathlib import Path
import sys
import torch

# D:\\Saito に合わせる。voxelmorph の親フォルダを指定する。
PROJECT_ROOT = Path(r'D:\\Saito')
if not (PROJECT_ROOT / 'voxelmorph').is_dir():
    raise FileNotFoundError(f'voxelmorph folder was not found under: {PROJECT_ROOT}')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dwt128_trilinear_curriculum import run_different_patient_finetuning

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
DATA_ROOT = PROJECT_ROOT / 'Data' / 'TestData'
PRETRAINED_CHECKPOINT = PROJECT_ROOT / '128dwt_trilinear_curriculum_checkpoints' / '128dwt_trilinear_curriculum_1to40_final.pth'
OUTPUT_DIR = PROJECT_ROOT / '128dwt_trilinear_different_patients_finetune_checkpoints'

TOTAL_EPOCHS = 30_000
STAGE_EPOCHS = 2_000
VISUALIZATION_EVERY = 2_000  # Moving / Fixed / Moved を表示・保存する間隔（epoch）
RUN_TRAINING = False  # 内容を確認後、True にしてこのセルを再実行する

print('device:', device)
print('data:', DATA_ROOT.resolve())
print('pretrained checkpoint:', PRETRAINED_CHECKPOINT.resolve())
print('output:', OUTPUT_DIR.resolve())

if RUN_TRAINING:
    model, final_path, history = run_different_patient_finetuning(
        data_root=DATA_ROOT,
        pretrained_checkpoint=PRETRAINED_CHECKPOINT,
        output_dir=OUTPUT_DIR,
        total_epochs=TOTAL_EPOCHS,
        stage_epochs=STAGE_EPOCHS,
        batch_size=2,
        learning_rate=1e-6,
        visualization_every=VISUALIZATION_EVERY,
        device=device,
    )
    print('final checkpoint:', final_path.resolve())
else:
    print('RUN_TRAINING=False: 設定確認のみ。True にしてから実行する。')
